In [42]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

In [43]:
# compute multimodal character continuity metrics (per-character profiles)
# for each character in each story: mcc_raw_char from text-image continuity, then MCC_char = tanh(mcc_raw_char)
# support both missing policies: zero and nan

MODELS = ['human', 'claude45', 'gpt4o', 'internvl3', 'llama4scout', 'qwen3vl']
PROMPTS = ['original', 'large']

COREF_DIR = Path('/mimer/NOBACKUP/groups/naiss2025-22-1187/coherence-driven-humans/models/linkappend/data-out/conll_to_json')
CHARACTERS_FILE = Path('/mimer/NOBACKUP/groups/naiss2025-22-1187/coherence-driven-humans/data/sampled_60/sampled_60_stories.json')
CHARACTER_MAPPINGS_FILE = Path('/mimer/NOBACKUP/groups/naiss2025-22-1187/coherence-tacl/data/sampled_60/character_mappings.json')

In [44]:
def load_jsonlines(filepath):
    documents = []
    with open(filepath, 'r') as f:
        for line in f:
            documents.append(json.loads(line))
    return documents

def clean_human_original_data(doc):
    if len(doc['sentences']) > 0:
        last_sent = doc['sentences'][-1]
        if len(last_sent) >= 3 and last_sent[-3:] == ['[', 'SENT', ']']:
            doc['sentences'][-1] = last_sent[:-3]
            num_tokens_before_last = sum(len(sent) for sent in doc['sentences'][:-1])
            removed_token_start = num_tokens_before_last + len(last_sent) - 3
            new_clusters = []
            for cluster in doc['clusters']:
                new_mentions = [m for m in cluster if m[0] < removed_token_start]
                if new_mentions:
                    new_clusters.append(new_mentions)
            doc['clusters'] = new_clusters
    return doc

def extract_story_id(doc_key):
    if isinstance(doc_key, int):
        return doc_key
    s = str(doc_key).replace('story_', '')
    parts = s.split('_')
    if len(parts) >= 2 and parts[0] == 'doc':
        s = parts[1]
    try:
        return int(float(s))
    except (ValueError, TypeError):
        return None

def load_character_stories(filepath):
    story_characters = {}
    valid_story_ids = set()
    with open(filepath, 'r') as f:
        for line in f:
            entry = json.loads(line)
            story_id = entry['story_id']
            characters = entry.get('characters', [])
            char_names = set()
            for char in characters:
                if isinstance(char, dict) and 'name' in char:
                    name = char['name']
                    if name and name != '{}':
                        char_names.add(name.lower())
                elif isinstance(char, str) and char and char != '{}':
                    char_names.add(char.lower())
            story_characters[story_id] = char_names
            if len(char_names) > 0:
                valid_story_ids.add(story_id)
    return story_characters, valid_story_ids

def load_character_mappings(filepath):
    with open(filepath, 'r') as f:
        raw_mappings = json.load(f)
    mappings = {}
    for story_id_str, images in raw_mappings.items():
        story_id = int(story_id_str)
        image_chars = []
        for img_name, chars in images.items():
            idx = int(img_name.split('_img')[1].split('.')[0])
            names = {c['name'].lower() for c in chars if 'name' in c and c['name']}
            image_chars.append((idx, names))
        image_chars.sort(key=lambda x: x[0])
        mappings[story_id] = [chars for _, chars in image_chars]
    return mappings

In [45]:
def get_sentence_boundaries(doc):
    boundaries = []
    current = 0
    for sent in doc.get('sentences', []):
        boundaries.append((current, current + len(sent) - 1))
        current += len(sent)
    return boundaries

def get_characters_per_sentence(doc, valid_characters):
    sentences = doc.get('sentences', [])
    clusters = doc.get('clusters', [])
    boundaries = get_sentence_boundaries(doc)
    chars_per_sentence = [set() for _ in range(len(sentences))]
    all_tokens = [tok for sent in sentences for tok in sent]

    for cluster in clusters:
        cluster_texts = set()
        for mention in cluster:
            start, end = mention[0], mention[1]
            if start < len(all_tokens) and end < len(all_tokens):
                span_text = ' '.join(all_tokens[start:end+1]).lower()
                cluster_texts.add(span_text)
                for tok in all_tokens[start:end+1]:
                    cluster_texts.add(tok.lower())

        matched_char = None
        for char_name in valid_characters:
            if char_name in cluster_texts:
                matched_char = char_name
                break

        if matched_char:
            for mention in cluster:
                start_token = mention[0]
                for sent_idx, (sent_start, sent_end) in enumerate(boundaries):
                    if sent_start <= start_token <= sent_end:
                        chars_per_sentence[sent_idx].add(matched_char)
                        break

    return chars_per_sentence

def compute_span_continuity(presence_list):
    if not any(presence_list):
        return None
    first_idx = last_idx = None
    for i, present in enumerate(presence_list):
        if present:
            if first_idx is None:
                first_idx = i
            last_idx = i
    if first_idx is None:
        return None
    span_len = last_idx - first_idx + 1
    appearances = sum(1 for i in range(first_idx, last_idx + 1) if presence_list[i])
    return appearances / span_len

def compute_mcc_character_rows_for_story(doc, story_id, valid_characters, char_mappings, missing_policy='zero'):
    if story_id not in char_mappings or not valid_characters:
        return []
    chars_per_sentence = get_characters_per_sentence(doc, valid_characters)
    image_chars = char_mappings.get(story_id, [])
    if len(chars_per_sentence) == 0 or len(image_chars) == 0:
        return []

    rows = []
    for char in sorted(valid_characters):
        text_presence = [char in chars_per_sentence[i] for i in range(len(chars_per_sentence))]
        tc = compute_span_continuity(text_presence)

        image_presence = [char in image_chars[i] for i in range(len(image_chars))]
        ic = compute_span_continuity(image_presence)

        if tc is not None and ic is not None:
            mcc_raw_char = float(1 - abs(tc - ic))
            mcc_char = float(np.tanh(mcc_raw_char))
            missing_reason = ''
        elif tc is not None or ic is not None:
            mcc_raw_char = 0.0
            mcc_char = float(np.tanh(mcc_raw_char))
            missing_reason = '__ONE_SIDE_MISSING__'
        else:
            fill = 0.0 if missing_policy == 'zero' else np.nan
            mcc_raw_char = fill
            mcc_char = fill
            missing_reason = '__NO_SPAN_MATCH__'

        rows.append({
            'character_name': char,
            'tc': tc if tc is not None else np.nan,
            'ic': ic if ic is not None else np.nan,
            'mcc_raw_char': mcc_raw_char,
            'MCC_char': mcc_char,
            'missing_reason': missing_reason
        })

    return rows

In [46]:
# load mcc data with missing-policy handling and prepare per-character output

def load_mcc_data(missing_policy='zero'):
    if missing_policy not in {'zero', 'nan'}:
        raise ValueError("missing_policy must be 'zero' or 'nan'")

    story_characters, valid_story_ids = load_character_stories(CHARACTERS_FILE)
    char_mappings = load_character_mappings(CHARACTER_MAPPINGS_FILE)
    rows = []

    for model in MODELS:
        for prompt in PROMPTS:
            seeds_to_load = ['seed42', 'seed43', 'seed44'] if (model == 'human' and prompt == 'large') else ['seed42']

            for seed in seeds_to_load:
                filepath = COREF_DIR / f"{model}_{prompt}_{seed}_test_snapshots__local_json-nopound_pred.jsonlines"
                if not filepath.exists():
                    continue
                docs = load_jsonlines(filepath)

                for doc in docs:
                    if model == 'human' and prompt == 'original':
                        doc = clean_human_original_data(doc)

                    story_id = extract_story_id(doc.get('doc_key'))
                    valid_chars = story_characters.get(story_id, set())

                    if story_id not in valid_story_ids or len(valid_chars) == 0:
                        fill = 0.0 if missing_policy == 'zero' else np.nan
                        rows.append({
                            'model': model, 'prompt': prompt, 'seed': seed, 'story_id': story_id,
                            'character_name': '__NO_ANNOTATION__',
                            'tc': np.nan, 'ic': np.nan,
                            'mcc_raw_char': fill, 'MCC_char': fill, 'missing_reason': '__NO_ANNOTATION__'
                        })
                        continue

                    char_rows = compute_mcc_character_rows_for_story(
                        doc, story_id, valid_chars, char_mappings, missing_policy=missing_policy
                    )

                    if len(char_rows) == 0:
                        fill = 0.0 if missing_policy == 'zero' else np.nan
                        rows.append({
                            'model': model, 'prompt': prompt, 'seed': seed, 'story_id': story_id,
                            'character_name': '__NO_MCC_MATCH__',
                            'tc': np.nan, 'ic': np.nan,
                            'mcc_raw_char': fill, 'MCC_char': fill, 'missing_reason': '__NO_MCC_MATCH__'
                        })
                        continue

                    for row in char_rows:
                        rows.append({
                            'model': model, 'prompt': prompt, 'seed': seed, 'story_id': story_id, **row
                        })

    return pd.DataFrame(rows)

def prepare_mcc_data(df):
    # per-character output (human large averaged across seeds by story_id + character_name)
    out = []
    for model in MODELS:
        for prompt in PROMPTS:
            df_mp = df[(df['model'] == model) & (df['prompt'] == prompt)]
            if len(df_mp) == 0:
                continue
            if model == 'human' and prompt == 'large':
                df_avg = (
                    df_mp.groupby(['story_id', 'character_name'])[['tc', 'ic', 'mcc_raw_char', 'MCC_char']]
                    .mean()
                    .reset_index()
                )
                for _, row in df_avg.iterrows():
                    out.append({
                        'model': model, 'prompt': prompt,
                        'story_id': int(row['story_id']), 'character_name': row['character_name'],
                        'tc': row['tc'], 'ic': row['ic'],
                        'mcc_raw_char': row['mcc_raw_char'], 'MCC_char': row['MCC_char']
                    })
            else:
                df_seed42 = df_mp[df_mp['seed'] == 'seed42']
                for _, row in df_seed42.iterrows():
                    out.append({
                        'model': model, 'prompt': prompt,
                        'story_id': int(row['story_id']), 'character_name': row['character_name'],
                        'tc': row['tc'], 'ic': row['ic'],
                        'mcc_raw_char': row['mcc_raw_char'], 'MCC_char': row['MCC_char']
                    })
    return pd.DataFrame(out)

In [47]:
## per story: compute mcc_raw from text-image character continuity, then squash with tanh
# mcc_raw is the average, across characters, of (1 - |tc - ic|) where:
#   tc = text span continuity (how continuously the character appears across the text span)
#   ic = image span continuity (how continuously the character appears across image sequence)
# MCC = tanh(mcc_raw) keeps values bounded in (0, 1) for easier comparison
#
# higher MCC -> stronger alignment between text-side and image-side character continuity
# lower MCC  -> weaker alignment / larger mismatch between text and image continuity patterns
#
# policy effect:
#   zero policy includes missing stories as 0.0 (penalizes missingness)
#   nan policy excludes missing stories from means (focuses on observed matches only)

In [48]:
df_mcc_raw = load_mcc_data(missing_policy='zero')
df_mcc = prepare_mcc_data(df_mcc_raw)

mcc_agg = (
    df_mcc.groupby(['model', 'prompt'])
    .agg(
        tc_mean=('tc', 'mean'),
        tc_std=('tc', 'std'),
        ic_mean=('ic', 'mean'),
        ic_std=('ic', 'std'),
        mcc_raw_mean=('mcc_raw_char', 'mean'),
        MCC_mean=('MCC_char', 'mean'),
        MCC_std=('MCC_char', 'std'),
        observed_rows=('MCC_char', 'count'),
        total_rows=('MCC_char', 'size')
    )
    .reset_index()
    .assign(missing_policy='zero', unit='character')
)

mcc_agg

,model,prompt,tc_mean,tc_std,ic_mean,ic_std,mcc_raw_mean,MCC_mean,MCC_std,observed_rows,total_rows,missing_policy,unit
0,claude45,large,0.845820,0.191302,0.840222,0.191087,0.583251,0.462374,0.338560,149,149,zero,character
1,claude45,original,0.842709,0.196232,0.840222,0.191087,0.580180,0.460620,0.337681,149,149,zero,character
2,gpt4o,large,0.847147,0.197645,0.840222,0.191087,0.675807,0.532118,0.315960,149,149,zero,character
3,gpt4o,original,0.874483,0.192476,0.840222,0.191087,0.634662,0.507711,0.310077,149,149,zero,character
4,human,large,0.893472,0.138221,0.840222,0.191087,0.532064,0.426774,0.265257,149,149,zero,character
5,human,original,0.863402,0.179484,0.840222,0.191087,0.530609,0.423483,0.343331,149,149,zero,character
6,internvl3,large,0.784237,0.209607,0.840222,0.191087,0.549193,0.449784,0.309454,149,149,zero,character
7,internvl3,original,0.806870,0.218423,0.840222,0.191087,0.540692,0.444294,0.308945,149,149,zero,character
8,llama4scout,large,0.841963,0.192359,0.840222,0.191087,0.287769,0.234841,0.321610,149,149,zero,character
9,llama4scout,original,0.937724,0.124993,0.840222,0.191087,0.162485,0.132386,0.268172,149,149,zero,character


In [49]:
# humans do not seem to align mentions of characters in text and their appearances in images
# (this does not really align with groovist results, which focuses on noun phrases)
# models seem to be aligning mentions between the modalities more
# standard deviation is generally large, so per-character behaviour is spread out

# formula check: mcc_raw_char = 1 - |tc - ic|, where tc and ic are continuity ratios in [0, 1]
# so |tc - ic| is also in [0, 1], meaning mcc_raw_char stays in [0, 1] and drops as tc/ic diverge
# then MCC_char = tanh(mcc_raw_char), still monotonic: larger mismatch -> lower MCC_char

# higher tc means text tends to keep referring to the same character continuously
# humans can have high tc (strong textual continuity), but lower MCC means text continuity does not match image continuity under this metric
# llama4scout is a very clear case of that: very high tc, but super low MCC

# the current metric directly measures closeness of tc and ic

In [50]:
df_mcc_raw = load_mcc_data(missing_policy='nan')
df_mcc = prepare_mcc_data(df_mcc_raw)

mcc_agg = (
    df_mcc.groupby(['model', 'prompt'])
    .agg(
        tc_mean=('tc', 'mean'),
        tc_std=('tc', 'std'),
        ic_mean=('ic', 'mean'),
        ic_std=('ic', 'std'),
        mcc_raw_mean=('mcc_raw_char', 'mean'),
        MCC_mean=('MCC_char', 'mean'),
        MCC_std=('MCC_char', 'std'),
        observed_rows=('MCC_char', 'count'),
        total_rows=('MCC_char', 'size')
    )
    .reset_index()
    .assign(missing_policy='nan', unit='character')
)

mcc_agg

,model,prompt,tc_mean,tc_std,ic_mean,ic_std,mcc_raw_mean,MCC_mean,MCC_std,observed_rows,total_rows,missing_policy,unit
0,claude45,large,0.845820,0.191302,0.840222,0.191087,0.689717,0.546775,0.298656,126,149,nan,character
1,claude45,original,0.842709,0.196232,0.840222,0.191087,0.691575,0.549059,0.295209,125,149,nan,character
2,gpt4o,large,0.847147,0.197645,0.840222,0.191087,0.799169,0.629250,0.237887,126,149,nan,character
3,gpt4o,original,0.874483,0.192476,0.840222,0.191087,0.744604,0.595661,0.245216,127,149,nan,character
4,human,large,0.893472,0.138221,0.840222,0.191087,0.634220,0.508715,0.204839,125,149,nan,character
5,human,original,0.863402,0.179484,0.840222,0.191087,0.642770,0.513000,0.310964,123,149,nan,character
6,internvl3,large,0.784237,0.209607,0.840222,0.191087,0.639295,0.523577,0.269548,128,149,nan,character
7,internvl3,original,0.806870,0.218423,0.840222,0.191087,0.634355,0.521258,0.267723,127,149,nan,character
8,llama4scout,large,0.841963,0.192359,0.840222,0.191087,0.337619,0.275523,0.331957,127,149,nan,character
9,llama4scout,original,0.937724,0.124993,0.840222,0.191087,0.196832,0.160370,0.287607,123,149,nan,character


In [51]:
out_dir = Path('./analysis_data/mcc')
out_dir.mkdir(parents=True, exist_ok=True)

for policy in ['zero', 'nan']:
    df_mcc_raw = load_mcc_data(missing_policy=policy)
    df_mcc = prepare_mcc_data(df_mcc_raw)

    mcc_agg = (
        df_mcc.groupby(['model', 'prompt'])
        .agg(
            tc_mean=('tc', 'mean'),
            tc_std=('tc', 'std'),
            ic_mean=('ic', 'mean'),
            ic_std=('ic', 'std'),
            mcc_raw_mean=('mcc_raw_char', 'mean'),
            MCC_mean=('MCC_char', 'mean'),
            MCC_std=('MCC_char', 'std'),
            observed_rows=('MCC_char', 'count'),
            total_rows=('MCC_char', 'size')
        )
        .reset_index()
        .assign(missing_policy=policy, unit='character')
    )

    df_mcc_raw.to_csv(out_dir / f'mcc_metrics_raw_{policy}_policy.csv', index=False)
    df_mcc.to_csv(out_dir / f'mcc_metrics_per_character_{policy}_policy.csv', index=False)
    mcc_agg.to_csv(out_dir / f'mcc_metrics_agg_per_character_{policy}_policy.csv', index=False)

print('Saved MCC outputs for policies: zero, nan')

Saved MCC outputs for policies: zero, nan
